# 06 — Frozen detector-to-Qwen submission

**Goal.** We build a reproducible submission using the trained DocLayout-YOLO detector and a Qwen configuration selected from validation results.

**Inputs.** Existing `src/work` artifacts, local detector weights, completed validation evaluations, the official sample submission, and test images. A fresh kernel avoids retaining training models in GPU memory.

**Evaluation scope.** This replaces the earlier manual test-repair workflow. Previous test inspection cannot be undone: this is a reproducible revised experiment, not a newly untouched test benchmark. Model selection uses validation only. The fixed empty-text policy for generation limits is an engineering rule inherited from debugging, not a proven quality improvement. Independent validation of that rule remains necessary.

**Conclusion.** To be completed after the run; file validity alone does not establish OCR quality.

In [ ]:
import csv
import gc
import importlib.metadata as metadata
import json
import sys
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from PIL import Image

CWD = Path.cwd().resolve()
CANDIDATES = [candidate for parent in (CWD, *CWD.parents) for candidate in (parent, parent / "src")]
PACKAGE = next(
    (
        p
        for p in CANDIDATES
        if (p / "rukopys" / "submission.py").is_file() and (p / "work").is_dir()
    ),
    None,
)

if PACKAGE is None:
    raise FileNotFoundError(
        "Open within the project; place the updated rukopys package " "in src next to work."
    )

if str(PACKAGE) not in sys.path:
    sys.path.insert(0, str(PACKAGE))

from rukopys.config import CLASS_PROMPT, PROMPTS
from rukopys.inference import detect_pages, write_submission
from rukopys.models import load_qwen
from rukopys.submission import transcribe_frozen
from rukopys.utils import atomic_json, digest, file_digest, read_json, valid_bbox

ROOT = (PACKAGE / "work").resolve()
WEIGHTS = PACKAGE / "weights" / "best.pt"
SAMPLE_SUBMISSION = ROOT / "data" / "sample_submission.csv"
TEST_DIR = ROOT / "data" / "kaggle_test" / "test" / "images"
RUN_NAME = "multitask_smoke"
DOWNLOAD_TEST_IMAGES = False
CONFIDENCE = 0.25
IMAGE_SIZE = 1024
MAX_TOKENS = 512
EXPECTED_WEIGHT_SHA256 = "2c4f64810fe020deb7d19e13513d8c588cdbe73a7f762503125e35eb9a110b65"

print("Source directory:", PACKAGE.name)
print("Work directory:", ROOT.relative_to(PACKAGE))

Source directory: src
Work directory: work


## Select the model on validation and lock the policy

We compare the completed base and adapter evaluations using routed corpus raw CER, choosing the base model on a tie. Only the original prompts and generation settings are eligible, and all other protocol fields must agree. Test predictions and previous repair files do not enter this decision.

Before opening test images, the selected model, prompts, decoding budget, detector settings, and failure rules are recorded in a lock file. Changed settings produce a different policy directory. The record is retained, and the policy stays fixed after test outputs become available.

A token-limit answer is exported as empty text while retaining its bounding box and predicted type. An invalid type stops the run rather than receiving a guessed label. No model training, automatic anti-repetition tuning, crop previews, or per-image exceptions are included.

In [ ]:
project = read_json(ROOT / "project.json")
validation = read_json(ROOT / "data" / "validation.json")
base_path = ROOT / "outputs" / "baseline.json"
adapter_path = ROOT / "outputs" / f"{RUN_NAME}_validation.json"
base = read_json(base_path)
adapted = read_json(adapter_path)
checkpoint = Path(read_json(ROOT / "runs" / RUN_NAME / "latest.json")["checkpoint"])

if not (checkpoint / "COMPLETE").is_file():
    raise ValueError("Adapter checkpoint is incomplete.")

for record in (base, adapted):
    if not record.get("complete") or "summary" not in record:
        raise ValueError("Complete both validation evaluations in notebook 04 first.")

    cfg = record["config"]

    if cfg["manifest"] != digest(validation) or [r["id"] for r in record["results"]] != [
        r["id"] for r in validation
    ]:
        raise ValueError("Validation data/order changed.")

    if (
        cfg["prompts"] != PROMPTS
        or cfg["class_prompt"] != CLASS_PROMPT
        or cfg["max_tokens"] != MAX_TOKENS
    ):
        raise ValueError("Validation prompts or generation budget differ.")

    if (
        cfg["protocol"]["model_id"] != project["model_id"]
        or cfg["protocol"]["model_revision"] != project["model_revision"]
    ):
        raise ValueError("Project model differs from validation.")

for key in (
    "model_id",
    "model_revision",
    "quantized",
    "dtype",
    "max_pixels",
    "packages",
):
    if base["config"]["protocol"][key] != adapted["config"]["protocol"][key]:
        raise ValueError(f"Validation protocol mismatch: {key}")

if base["config"]["protocol"]["adapter_sha256"] is not None:
    raise ValueError("Baseline is not the base model.")

if adapted["config"]["protocol"]["adapter_sha256"] != file_digest(
    checkpoint / "adapter_model.safetensors"
):
    raise ValueError("Adapter weights differ from their validation evaluation.")

base_score = base["summary"]["routed"]["corpus_raw_cer"]
adapter_score = adapted["summary"]["routed"]["corpus_raw_cer"]

if base_score is None or adapter_score is None:
    raise ValueError("Validation scores are missing.")

SELECTED_MODE = "base" if base_score <= adapter_score else "adapter"
SELECTED_PROTOCOL = (base if SELECTED_MODE == "base" else adapted)["config"]["protocol"]
SELECTED_CHECKPOINT = None if SELECTED_MODE == "base" else checkpoint

if not WEIGHTS.is_file() or file_digest(WEIGHTS) != EXPECTED_WEIGHT_SHA256:
    raise ValueError("Place the supplied detector weights at src/weights/best.pt.")

POLICY = {
    "schema": "validation_selected_v1",
    "selection_metric": "routed_corpus_raw_cer",
    "validation_scores": {
        "base": base_score,
        "adapter": adapter_score,
    },
    "validation_evidence": {
        "base_sha256": file_digest(base_path),
        "adapter_sha256": file_digest(adapter_path),
    },
    "selected_mode": SELECTED_MODE,
    "protocol": SELECTED_PROTOCOL,
    "prompts": dict(PROMPTS),
    "class_prompt": CLASS_PROMPT,
    "max_tokens": MAX_TOKENS,
    "decoding": "existing greedy generate; no per-test tuning",
    "unresolved_text": "empty",
    "invalid_type": "stop",
    "detector_weights_sha256": EXPECTED_WEIGHT_SHA256,
    "confidence": CONFIDENCE,
    "image_size": IMAGE_SIZE,
    "historical_limitation": (
        "This test set was inspected in an earlier experiment; " "not an untouched holdout."
    ),
    "implementation_sha256": {
        name: file_digest(PACKAGE / "rukopys" / name)
        for name in ("models.py", "inference.py", "submission.py")
    },
}

POLICY_DIR = ROOT / "outputs" / "frozen_submission" / digest(POLICY)
POLICY_DIR.mkdir(parents=True, exist_ok=True)
lock = POLICY_DIR / "policy.json"

if lock.exists():
    if read_json(lock) != POLICY:
        raise ValueError("Saved policy differs.")
else:
    atomic_json(lock, POLICY)

print("Validation-selected model:", SELECTED_MODE)
print("Policy saved:", lock.relative_to(PACKAGE))

Validation-selected model: base
Policy saved: work\outputs\frozen_submission\4895645b850e377859ab34b4bc625daddd46df36176d48cc7a24531da72e98dc\policy.json


## Check the test inputs

We use the official sample submission for filenames only; region contents and test annotations are not read. The optional download requests only test images at the recorded dataset revision. Existing local images and detection results are reused when their fingerprints agree.

Detector confidence remains fixed rather than being tuned here. The detector and Qwen training splits differed, so this notebook does not establish a clean integrated validation score.

In [ ]:
if DOWNLOAD_TEST_IMAGES:
    snapshot_download(
        repo_id=project["dataset_id"],
        repo_type="dataset",
        revision=project["dataset_revision"],
        allow_patterns=["test/images/*"],
        local_dir=str(ROOT / "data" / "kaggle_test"),
        max_workers=4,
    )

if not SAMPLE_SUBMISSION.is_file() or not TEST_DIR.is_dir():
    raise FileNotFoundError("Provide work/data/sample_submission.csv and local test images.")

with SAMPLE_SUBMISSION.open(encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    if reader.fieldnames != ["image", "regions"]:
        raise ValueError("Unexpected official submission columns.")

    EXPECTED_IMAGES = [row["image"] for row in reader]

if not EXPECTED_IMAGES or len(EXPECTED_IMAGES) != len(set(EXPECTED_IMAGES)):
    raise ValueError("Empty or duplicate official image list.")

files_by_name = {}

for path in TEST_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in (
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".tif",
        ".tiff",
    ):
        files_by_name.setdefault(path.name, []).append(path)

PAGES = []

for name in EXPECTED_IMAGES:
    matches = files_by_name.get(Path(name).name, [])

    if len(matches) != 1:
        raise ValueError(f"Expected exactly one image for {name}.")

    path = matches[0].resolve()

    with Image.open(path) as image:
        width, height = image.size

    PAGES.append(
        {
            "image": name,
            "image_path": str(path),
            "width": width,
            "height": height,
        }
    )

TEST_SIGNATURE = digest(
    {
        "pages": [{**p, "sha256": file_digest(p["image_path"])} for p in PAGES],
        "weights": EXPECTED_WEIGHT_SHA256,
        "confidence": CONFIDENCE,
        "image_size": IMAGE_SIZE,
    }
)
DETECTIONS = ROOT / "outputs" / "test_detections" / TEST_SIGNATURE / "detections.json"
RUN_DIR = POLICY_DIR / TEST_SIGNATURE
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = RUN_DIR / "predictions.json"
SUBMISSION = RUN_DIR / "submission.csv"

print("Test page count:", len(PAGES))

Test page count: 385


## Run inference

We load complete compatible predictions without loading either GPU model. Incomplete detection and transcription continue from saved pages; only an unfinished page is repeated after interruption. Old `kaggle_test` repair files are neither read nor overwritten.

New inference requires the existing CUDA environment and `doclayout-yolo`. Dependency installation remains separate from this notebook. Predictions follow the locked policy without using test text to choose settings.

In [4]:
complete_record = read_json(OUTPUT) if OUTPUT.is_file() else None
USE_SAVED_RESULTS = bool(complete_record and complete_record.get("complete"))

if USE_SAVED_RESULTS:
    cfg = complete_record["config"]

    if cfg["policy"] != POLICY or cfg["protocol"] != SELECTED_PROTOCOL:
        raise ValueError("Saved inference policy differs.")

    if cfg["image_sha256"] != {p["image"]: file_digest(p["image_path"]) for p in PAGES}:
        raise ValueError("Test images changed.")

    detection_record = read_json(DETECTIONS)
    expected_detection = {
        "weights_sha256": EXPECTED_WEIGHT_SHA256,
        "pages_sha256": digest(PAGES),
        "confidence": CONFIDENCE,
        "image_size": IMAGE_SIZE,
    }

    if detection_record["config"] != expected_detection or not detection_record.get("complete"):
        raise ValueError("Detection cache differs or is incomplete.")

    if cfg["detections_sha256"] != digest(detection_record["pages"]):
        raise ValueError("Saved detections changed.")

    result = complete_record["pages"]
    print("Reusing completed inference; GPU model loading skipped.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA-enabled kernel is required for pending inference.")

    expected_detection = {
        "weights_sha256": EXPECTED_WEIGHT_SHA256,
        "pages_sha256": digest(PAGES),
        "confidence": CONFIDENCE,
        "image_size": IMAGE_SIZE,
    }
    detection_record = read_json(DETECTIONS) if DETECTIONS.is_file() else None

    if detection_record and detection_record["config"] != expected_detection:
        raise ValueError("Detection cache uses different settings.")

    if detection_record and detection_record.get("complete"):
        pages = detection_record["pages"]
    else:
        pages = detect_pages(
            PAGES,
            WEIGHTS,
            DETECTIONS,
            confidence=CONFIDENCE,
            image_size=IMAGE_SIZE,
        )

    if [p["image"] for p in pages] != EXPECTED_IMAGES:
        raise ValueError("Detector page coverage differs.")

    gc.collect()
    torch.cuda.empty_cache()

    model, processor, protocol = load_qwen(
        ROOT,
        adapter=SELECTED_CHECKPOINT,
        quantized=True,
        max_pixels=SELECTED_PROTOCOL["max_pixels"],
    )

    if protocol != SELECTED_PROTOCOL:
        raise ValueError("Runtime differs from the selected validation protocol.")

    result = transcribe_frozen(
        pages,
        model,
        processor,
        protocol,
        OUTPUT,
        POLICY,
    )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1/385 pages saved
2/385 pages saved
3/385 pages saved
4/385 pages saved
5/385 pages saved
6/385 pages saved
7/385 pages saved
8/385 pages saved
9/385 pages saved
10/385 pages saved
11/385 pages saved
12/385 pages saved
13/385 pages saved
14/385 pages saved
15/385 pages saved
16/385 pages saved
17/385 pages saved
18/385 pages saved
19/385 pages saved
20/385 pages saved
21/385 pages saved
22/385 pages saved
23/385 pages saved
24/385 pages saved
25/385 pages saved
26/385 pages saved
27/385 pages saved
28/385 pages saved
29/385 pages saved
30/385 pages saved
31/385 pages saved
32/385 pages saved
33/385 pages saved
34/385 pages saved
35/385 pages saved
36/385 pages saved
37/385 pages saved
38/385 pages saved
39/385 pages saved
40/385 pages saved
41/385 pages saved
42/385 pages saved
43/385 pages saved
44/385 pages saved
45/385 pages saved
46/385 pages saved
47/385 pages saved
48/385 pages saved
49/385 pages saved
50/385 pages saved
51/385 pages saved
52/385 pages saved
53/385 pages saved
54

KeyboardInterrupt: 

## Audit and export the submission

We retain every bounding box and predicted type. Token-limit failures produce empty exported text and remain flagged in diagnostics. This is abstention rather than a corrected transcription; individual test outputs are not manually edited.

All expected pages must be present before export. The CSV and audit are written from the checked predictions, including when those predictions were loaded from a completed run.

In [ ]:
record = read_json(OUTPUT)

if not record.get("complete"):
    raise ValueError("Inference is incomplete.")

if [p["image"] for p in result] != EXPECTED_IMAGES:
    raise ValueError("Incomplete or reordered test predictions.")

dimensions = {p["image"]: (p["width"], p["height"]) for p in PAGES}
abstentions = 0

for page in result:
    if len(page["diagnostics"]) != len(page["regions"]):
        raise ValueError("Missing diagnostic records.")

    for index, diagnostic in enumerate(page["diagnostics"]):
        if diagnostic["region_index"] != index:
            raise ValueError("Diagnostic order mismatch.")

        region = page["regions"][index]
        valid_bbox(region["bbox"], *dimensions[page["image"]])

        if diagnostic["transcription"]["hit_token_limit"]:
            if region["text"] != "" or not diagnostic["abstained"]:
                raise ValueError("Unresolved text was not withheld.")

            abstentions += 1

# Truncation flags remain intact; all corresponding exported texts were checked empty
write_submission(result, EXPECTED_IMAGES, SUBMISSION, allow_truncated=True)

with SUBMISSION.open(encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    if reader.fieldnames != ["image", "regions"]:
        raise ValueError("CSV columns differ.")

    exported = list(reader)

if [r["image"] for r in exported] != EXPECTED_IMAGES:
    raise ValueError("CSV page coverage differs.")

for row in exported:
    if not isinstance(json.loads(row["regions"]), list):
        raise ValueError("Invalid region serialization.")

atomic_json(
    RUN_DIR / "export_audit.json",
    {
        "policy_sha256": digest(POLICY),
        "predictions_sha256": file_digest(OUTPUT),
        "csv_sha256": file_digest(SUBMISSION),
        "pages": len(result),
        "empty_text_abstentions": abstentions,
        "historical_test_inspection": True,
    },
)

print("Submission validation passed.")
print("Page count:", len(result))
print("Empty-text abstentions:", abstentions)
print("CSV for manual Kaggle upload:", SUBMISSION.resolve())

## Results and limitations

- Export status and aggregate counts: [complete after running]
- Kaggle submission identifier and score: [complete after upload]

We export a file for manual upload; this notebook does not submit it automatically. Keep the policy, predictions, and audit alongside the CSV. Completed outputs from notebooks 01–05 remain unchanged.

New fixed-policy transcription may require a full pass because old outputs used a different model or policy. Relabeling those outputs would not produce a valid new experiment.

The Kaggle test set was already inspected during an earlier debugging session. Removing previews prevents further manual feedback but cannot restore test independence. A new untouched holdout and a common detector/Qwen split are needed for a stronger generalization claim. Page separation alone does not guarantee writer or dictation-text independence.